# Analysis template — plot a training run

A reusable starting point for inspecting a run. `rl-lab train` writes a
`metrics.csv` under `runs/<run-name>/`; point `RUN_CSV` at it and run all cells.
Uses only the default stack (csv + numpy + matplotlib — no pandas).


In [ ]:
import csv
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


## 1. Load a metrics CSV

If the file does not exist yet, we fall back to a small synthetic curve so
this template runs top-to-bottom on a fresh checkout.


In [ ]:
RUN_CSV = Path('runs/example/metrics.csv')  # <- change to your run

def load_csv(path):
    """Read a metrics.csv into a dict of column-name -> float array."""
    with open(path, newline='') as fh:
        rows = list(csv.DictReader(fh))
    cols = {}
    for key in rows[0]:
        cols[key] = np.array([float(r[key]) for r in rows if r.get(key) not in (None, '')])
    return cols

if RUN_CSV.exists():
    data = load_csv(RUN_CSV)
    print('Loaded', RUN_CSV, 'columns:', list(data))
else:
    print(f'{RUN_CSV} not found — using synthetic data so the template still runs.')
    steps = np.arange(0, 200)
    rng = np.random.default_rng(0)
    ret = -1.0 + 1.0 * (1 - np.exp(-steps / 50.0)) + rng.normal(0, 0.05, steps.shape)
    data = {'step': steps.astype(float), 'episode_return': ret}


## 2. Plot the learning curve

Pick the x and y columns (defaults: `step` and `episode_return`).


In [ ]:
x_col = 'step' if 'step' in data else list(data)[0]
y_col = 'episode_return' if 'episode_return' in data else list(data)[-1]

def smooth(y, k=10):
    return np.convolve(y, np.ones(k) / k, mode='valid') if len(y) >= k else y

x, y = data[x_col], data[y_col]
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(x, y, alpha=0.3, label=y_col)
ys = smooth(y)
ax.plot(x[len(x) - len(ys):], ys, lw=2, label=f'{y_col} (smoothed)')
ax.set_xlabel(x_col); ax.set_ylabel(y_col); ax.legend(); ax.grid(alpha=0.3)
ax.set_title('Training curve')
plt.show()


## 3. Your analysis

Add cells below: compare runs, compute success rate over the last N episodes,
overlay several `metrics.csv` files, etc.
